# Calixto

Steps that are repeated in this notebook live in `link_prediction_utils.py` (keep it in the same folder):
`community_map`, `features` / `feature_matrix`, `draw_network`, `evaluate_model`, `search_summary`.
Everything that runs only once is written out below.

## Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import random
from sklearn.base import clone
from matplotlib.patches import Patch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (RandomForestClassifier, VotingClassifier,
                              StackingClassifier, GradientBoostingClassifier)

from link_prediction_utils import (community_map, features, feature_matrix,
                                   draw_network, evaluate_model, search_summary)

## Import

In [ ]:
raw_network = nx.read_edgelist("input_data/edges_train.edgelist", data=False, nodetype = int, delimiter=',')

In [ ]:
G = raw_network.copy()

In [ ]:
raw_attributes = pd.read_csv('input_data/attributes.csv')
raw_attributes.info()

In [ ]:
attrs = raw_attributes.copy()

# Attach the attribute to every node now, so the per-group statistics below work
G.add_nodes_from(attrs.ID)
nx.set_node_attributes(G, dict(zip(attrs.ID, attrs.attribute)), "attr")

## Analysis
### Network

In [ ]:
# Nodes, edges, density
N, E = G.number_of_nodes(), G.number_of_edges()
print(N, E, nx.density(G), nx.number_of_isolates(G))
lcc = G.subgraph(max(nx.connected_components(G), key=len))
print(nx.number_connected_components(G), lcc.number_of_nodes())

In [ ]:
# Degree distribution
deg = pd.Series(dict(G.degree()))
print(deg.describe())

In [ ]:
# Degree mean grouped by attribute
attr_s = pd.Series(nx.get_node_attributes(G, "attr"))
print(deg.groupby(attr_s).mean())

In [ ]:
# Clustering
C = nx.average_clustering(G)
print(C)

### Attributes

In [ ]:
pos = nx.spring_layout(G, seed=42)

In [ ]:
# Draw the original graph
draw_network(G, pos, node_size=5)
plt.show()

In [ ]:
# Draw the graph coloured by attribute
cats = sorted(attrs.attribute.unique())
cmap = plt.get_cmap("tab10")
color_of = {c: cmap(i) for i, c in enumerate(cats)}
node_colors = [color_of[G.nodes[n]["attr"]] for n in G.nodes()]

draw_network(G, pos, node_colors=node_colors, node_size=12)
plt.legend(handles=[Patch(color=color_of[c], label=c) for c in cats], title="attribute")
plt.axis("off"); plt.show()

In [ ]:
# Use modularity communities to segment the network
community_of = community_map(G, resolution=0.8)
print(len(set(community_of.values())), "communities")

node_colors = [community_of[node] for node in G.nodes()]
draw_network(G, pos, node_colors=node_colors, node_size=5)
plt.show()

### Link prediction

In [ ]:
RESOLUTION = 1.5        # community resolution for same_comm (MUST be the same for training and Kaggle)

# Remove 10% of the dataset
random.seed(42)

edges = list(G.edges())
nodes = list(G.nodes())

n_hide = int(0.1 * len(edges))
hidden = random.sample(edges, n_hide)

G_feat = G.copy()
G_feat.remove_edges_from(hidden)

neg = set()

while len(neg) < n_hide:
    u, v = random.sample(nodes, 2)
    if not G.has_edge(u, v):
        neg.add((min(u, v), max(u, v)))

# Communities computed on G_feat (not on G) → no leakage
comm_of = community_map(G_feat, resolution=RESOLUTION)

pairs  = hidden + list(neg)
Y      = np.array([1]*len(hidden) + [0]*len(neg))
X      = feature_matrix(G_feat, pairs, comm_of)

In [ ]:
# See relevance of features
pd.Series({f: roc_auc_score(Y, X[f]) for f in X.columns}).sort_values(ascending=False)

In [ ]:
# Search for best community parameter
for r in [0.5, 1.0, 1.5, 2.0, 3.0]:
    comm_r = community_map(G_feat, resolution=r)
    X_r = X.copy()
    X_r["same_comm"] = [int(comm_r[u] == comm_r[v]) for u, v in pairs]
    print(r, roc_auc_score(Y, X_r["same_comm"]))

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42)

#TODO: train - validation - test splits (explained via cross validation in following code)

### Models

In [ ]:
clf = LogisticRegression(random_state=0, C=2.0)
lr_scores = evaluate_model("Logistic regression", clf, X_train, y_train, X_test, y_test)

#TODO: include other metrics (confusion matrix) also do a gridsearch

In [ ]:
dtc = DecisionTreeClassifier(max_depth=2)
dt_scores = evaluate_model("Decision tree", dtc, X_train, y_train, X_test, y_test)

#TODO: make sure to include the ethics metrics

In [ ]:
# Random forest
models = {
    "Decision tree": DecisionTreeClassifier(max_depth=2, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
} #TODO: try different seeds

for name, model in models.items():
    evaluate_model(name, model, X_train, y_train, X_test, y_test)

# Which features does the forest rely on?
rf = models["Random forest"]
print(pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).round(3))

In [ ]:
for d in [2, 3, 5, 8, None]:
    dt = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=42), X_train, y_train, cv=5).mean()
    rf = cross_val_score(RandomForestClassifier(n_estimators=200, max_depth=d, random_state=42), X_train, y_train, cv=5).mean()
    print(f"depth={d}:  DT {dt:.3f} | RF {rf:.3f}")

In [ ]:
dt = DecisionTreeClassifier(max_depth=2, random_state=42)
rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)

models = {
    "Voting (DT+RF)":   VotingClassifier([("dt", dt), ("rf", rf)], voting="soft"),
    "Stacking (DT+RF)": StackingClassifier([("dt", dt), ("rf", rf)],
                                           final_estimator=LogisticRegression()),
    "Boosting":         GradientBoostingClassifier(n_estimators=100, max_depth=2, random_state=42),
}

for name, model in models.items():
    evaluate_model(name, model, X_train, y_train)      # CV only

### Testing Hyperparameters

In [ ]:
# Decision tree
dt_grid = {
    "max_depth": [1, 2, 3, 4, 5, 8, None],
    "min_samples_leaf": [1, 5, 20, 50],
    "criterion": ["gini", "entropy"],
}
dt_search = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_grid, cv=5, scoring="accuracy")
dt_search.fit(X_train, y_train)

# Logistic regression (scaler inside the pipeline)
lr_pipe = make_pipeline(StandardScaler(), LogisticRegression(solver="saga", max_iter=5000))
lr_grid = {
    "logisticregression__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "logisticregression__l1_ratio": [0, 0.5, 1],     # L2, elastic net, L1
}
lr_search = GridSearchCV(lr_pipe, lr_grid, cv=5, scoring="accuracy")
lr_search.fit(X_train, y_train)

for name, s in [("Decision tree", dt_search), ("Logistic regression", lr_search)]:
    summary = search_summary(s, X_test, y_test)
    print(f"{name}: {summary['best_params']}")
    print(f"   CV {summary['cv_mean']:.3f} ± {summary['cv_std']:.3f} | Test {summary['test_acc']:.3f}")

## Kaggle

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. DECLARE THE FINAL MODEL
#    Both models tied in CV (difference << std), so we choose the
#    decision tree for interpretability (see report: Model selection).
# ─────────────────────────────────────────────────────────────
chosen_search = dt_search            # switch to lr_search to use logistic regression instead
chosen_name   = "Decision tree"

summary = search_summary(chosen_search, X_test, y_test)

print("=" * 60)
print(f"FINAL MODEL: {chosen_name}")
for k, v in summary["best_params"].items():
    print(f"   {k:20s} = {v}")
print(f"   CV accuracy (train) = {summary['cv_mean']:.3f} ± {summary['cv_std']:.3f}")
print(f"   Test accuracy       = {summary['test_acc']:.3f}")
print("=" * 60)

# ─────────────────────────────────────────────────────────────
# 2. REFIT on ALL labelled data (train + test) with these hyperparameters
# ─────────────────────────────────────────────────────────────
final_model = clone(chosen_search.best_estimator_)   # same settings, untrained copy
final_model.fit(X, Y)

# ─────────────────────────────────────────────────────────────
# 3. FEATURES FOR solutionInput — computed on the FULL graph G
# ─────────────────────────────────────────────────────────────
solution = pd.read_csv("input_data/solutionInput.csv")              # ID, int1, int2
comm_G = community_map(G, resolution=RESOLUTION)

X_sol = feature_matrix(G, zip(solution.int1, solution.int2), comm_G)
X_sol = X_sol[X_train.columns]                                       # same columns & order as training

# ─────────────────────────────────────────────────────────────
# 4. PREDICT + SAVE
# ─────────────────────────────────────────────────────────────
solution["prediction"] = final_model.predict(X_sol)
print("Predicted positive rate:", round(solution.prediction.mean(), 3), "(should be ≈ 0.5)")
solution[["ID", "prediction"]].to_csv("output_kaggle/submission_calixto.csv", index=False)
print("Saved → output_kaggle/submission_calixto.csv")

In [ ]:
# Prediction for a single edge given as input
u, v = 164, 1010
x = feature_matrix(G, [(u, v)], comm_G)[X_train.columns]
print(f"Link {u}-{v}:", final_model.predict(x)[0])